# Step 9: GPT — Decoder-Only Transformer for Language Modeling

The Transformer in the previous notebook was an **encoder** — every token can attend to
every other token. For classification, that's fine. For **language modeling** (predicting
the next token), we can't let position t attend to position t+1 — that would leak the answer.

**GPT** (Radford et al., 2018) uses a **decoder-only** Transformer with **causal masking**:
position t can only attend to positions 0 through t. Combined with next-token prediction
as the pre-training objective, this gives a model that can:
- Generate coherent text by sampling autoregressively
- Learn from any raw text — no labels needed
- Acquire diverse capabilities as scale increases

**The objective**: `L = -Σₜ log p(xₜ | x₁, ..., xₜ₋₁)`

**What you'll see:**
1. Causal masking: how we prevent information leakage from the future
2. A full GPT-style model: token embedding + positional embedding + N decoder blocks
3. Training on a character-level corpus (same as notebooks 4-5 for comparison)
4. Autoregressive text generation with temperature sampling
5. How model scale affects generation quality

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Causal Masking

The only architectural change from the encoder: apply an upper-triangular mask
to the attention scores, setting future positions to `-inf` before softmax.
After softmax, those positions get weight 0 — effectively, they don't exist.

```
mask = [[1, 0, 0, 0],
         [1, 1, 0, 0],
         [1, 1, 1, 0],
         [1, 1, 1, 1]]

scores.masked_fill(mask == 0, -inf)
```

This allows all positions to be computed in parallel (during training),
while preserving the causal structure that prevents future leakage.

In [ ]:
# Visualize the causal mask
T = 6
mask = torch.tril(torch.ones(T, T))  # lower triangular

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(mask, cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('Causal mask\n(1=can attend, 0=blocked)')
for i in range(T):
    for j in range(T):
        axes[0].text(j, i, int(mask[i,j].item()), ha='center', va='center', fontsize=10)

# Fake attention scores before masking
scores = torch.randn(T, T)
scores_masked = scores.masked_fill(mask == 0, float('-inf'))
weights = F.softmax(scores_masked, dim=-1)

axes[1].imshow(scores_masked.detach().numpy(), cmap='RdBu_r')
axes[1].set_title('After masking: future = -inf')

axes[2].imshow(weights.detach().numpy(), cmap='Blues')
axes[2].set_title('After softmax: future has weight 0')

for ax in axes:
    ax.set_xlabel('Key position'); ax.set_ylabel('Query position')

plt.tight_layout()
plt.savefig('09_causal_mask.png', bbox_inches='tight')
plt.show()

## GPT Architecture

```
Input tokens: x₁, x₂, ..., xₜ
  ↓
Token embedding (vocab_size → d_model)       ← learned
 + Positional embedding (max_len → d_model)  ← also learned (not sinusoidal)
  ↓
N × Decoder Block:
  LayerNorm → CausalSelfAttention → Residual
  LayerNorm → FeedForward         → Residual   (pre-LN: normalize before sub-layer)
  ↓
LayerNorm → Linear(d_model → vocab_size) → Softmax
```

**Pre-LN** (used in GPT-2 and later): normalize **before** each sub-layer.
This makes training more stable than post-LN, especially at large scale.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_len, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads

        self.Wqkv = nn.Linear(d_model, 3 * d_model, bias=False)  # fused projection
        self.Wo   = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

        # Register causal mask as a buffer (not a parameter — it doesn't get gradients)
        causal = torch.tril(torch.ones(max_len, max_len)).view(1, 1, max_len, max_len)
        self.register_buffer('mask', causal)

    def forward(self, x):
        B, T, D = x.shape
        H, dk = self.n_heads, self.d_k

        # Fused Q, K, V projection
        qkv = self.Wqkv(x)                              # (B, T, 3D)
        Q, K, V = qkv.split(D, dim=-1)                  # each (B, T, D)

        # Reshape to (B, H, T, dk)
        Q = Q.view(B, T, H, dk).transpose(1, 2)
        K = K.view(B, T, H, dk).transpose(1, 2)
        V = V.view(B, T, H, dk).transpose(1, 2)

        # Scaled dot product attention with causal mask
        scores = Q @ K.transpose(-2, -1) / math.sqrt(dk)   # (B, H, T, T)
        scores = scores.masked_fill(self.mask[:,:,:T,:T] == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        weights = self.drop(weights)

        out = (weights @ V).transpose(1, 2).contiguous().view(B, T, D)  # (B, T, D)
        return self.Wo(out)


class GPTBlock(nn.Module):
    """Pre-LN Transformer decoder block."""
    def __init__(self, d_model, n_heads, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_len, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # attention with pre-LN
        x = x + self.ff(self.ln2(x))     # FFN with pre-LN
        return x


class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers,
                 max_len=256, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)   # learned (not sinusoidal)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.Sequential(*[
            GPTBlock(d_model, n_heads, d_ff, max_len, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f  = nn.LayerNorm(d_model)
        self.head  = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: token embedding = output projection matrix
        self.head.weight = self.tok_emb.weight

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, x):
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        x    = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        x    = self.blocks(x)
        x    = self.ln_f(x)
        return self.head(x)  # (B, T, vocab_size)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Autoregressive generation: repeatedly predict and append next token."""
        self.eval()
        max_len = self.pos_emb.num_embeddings
        for _ in range(max_new_tokens):
            # Crop context to max_len (GPT's attention is bounded)
            idx_cond = idx[:, -max_len:]
            logits = self(idx_cond)[:, -1, :]  # logits for the last position

            logits = logits / temperature

            if top_k is not None:
                # Keep only top-k logits, set rest to -inf
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, -1:]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx], dim=1)
        return idx


# Build two models: small and medium — same data, different capacity
configs = {
    'small':  dict(d_model=64,  n_heads=4, d_ff=256,  n_layers=2),
    'medium': dict(d_model=128, n_heads=8, d_ff=512,  n_layers=4),
}
for name, cfg in configs.items():
    m = GPT(vocab_size=50, max_len=128, dropout=0.0, **cfg)
    p = sum(p.numel() for p in m.parameters())
    print(f"GPT-{name}: {p:,} parameters")

In [ ]:
# Same corpus as notebooks 4 and 5 — direct comparison possible
text = """To be or not to be that is the question
Whether tis nobler in the mind to suffer
The slings and arrows of outrageous fortune
Or to take arms against a sea of troubles
And by opposing end them to die to sleep
No more and by a sleep to say we end
The heartache and the thousand natural shocks
That flesh is heir to tis a consummation
Devoutly to be wished to die to sleep
To sleep perchance to dream aye theres the rub
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil
Must give us pause there is the respect
That makes calamity of so long life"""

chars     = sorted(set(text))
VOCAB_SIZE = len(chars)
char2idx  = {c: i for i, c in enumerate(chars)}
idx2char  = {i: c for c, i in char2idx.items()}
data      = torch.tensor([char2idx[c] for c in text], dtype=torch.long)

print(f"Corpus: {len(text)} chars, {VOCAB_SIZE} unique")

# Build both models
BLOCK_LEN = 64  # context window
gpt_small  = GPT(VOCAB_SIZE, d_model=64,  n_heads=4, d_ff=256,  n_layers=2,
                 max_len=BLOCK_LEN, dropout=0.0).to(device)
gpt_medium = GPT(VOCAB_SIZE, d_model=128, n_heads=8, d_ff=512,  n_layers=4,
                 max_len=BLOCK_LEN, dropout=0.0).to(device)

def train_gpt(model, data, n_steps=1000, batch_size=8, lr=3e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    losses = []
    model.train()
    for step in range(n_steps):
        # Random batch of sequences
        starts = torch.randint(0, len(data) - BLOCK_LEN, (batch_size,))
        x = torch.stack([data[s:s+BLOCK_LEN]   for s in starts]).to(device)
        y = torch.stack([data[s+1:s+BLOCK_LEN+1] for s in starts]).to(device)

        logits = model(x)
        loss   = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
    return losses

print("Training GPT-small...")
losses_small  = train_gpt(gpt_small,  data, n_steps=1500)
print("Training GPT-medium...")
losses_medium = train_gpt(gpt_medium, data, n_steps=1500)
print("Done.")

In [ ]:
def smooth(a, w=30):
    return np.convolve(a, np.ones(w)/w, 'valid')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(smooth(losses_small),  'tomato',    lw=2, label='GPT-small')
ax.plot(smooth(losses_medium), 'steelblue', lw=2, label='GPT-medium')
ax.axhline(np.log(VOCAB_SIZE), color='k', linestyle=':', lw=1.5, label='random chance')
ax.set_xlabel('Training step'); ax.set_ylabel('Cross-entropy loss')
ax.set_title('GPT training: more parameters → lower loss (same data)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('09_gpt_training.png', bbox_inches='tight')
plt.show()

print(f"Final loss — small: {losses_small[-1]:.4f}, medium: {losses_medium[-1]:.4f}")

In [ ]:
# Compare generation quality at multiple temperatures
def decode(idx_tensor):
    return ''.join(idx2char[i] for i in idx_tensor[0].tolist())

seed = torch.tensor([[char2idx['T']]], device=device)

print("=" * 60)
print("GPT-SMALL SAMPLES")
print("=" * 60)
for temp in [0.5, 1.0, 1.5]:
    out = gpt_small.generate(seed, max_new_tokens=150, temperature=temp, top_k=10)
    print(f"\nTemperature {temp}:")
    print(decode(out))

print()
print("=" * 60)
print("GPT-MEDIUM SAMPLES")
print("=" * 60)
for temp in [0.5, 1.0, 1.5]:
    out = gpt_medium.generate(seed, max_new_tokens=150, temperature=temp, top_k=10)
    print(f"\nTemperature {temp}:")
    print(decode(out))

## Why This Scales

GPT's key insight: the same causal language modeling objective works for **any text**.
Given 10B tokens of internet text and a model with 175B parameters (GPT-3),
the model learns to:
- Complete code
- Answer questions
- Translate languages
- Solve math problems

None of these were explicitly trained for — they emerge from predicting the next token
across text that contains examples of all of them.

**Scaling laws** (from the `scaling-laws` wiki page) tell us:
```
L(N) ∝ N^{-0.076}   — loss improves predictably with model size
L(D) ∝ D^{-0.095}   — and with data size
L(C) ∝ C^{-0.050}   — and with compute
```

Smooth, power-law improvement — no sudden wall. Just keep scaling.

In [ ]:
# Demonstrate top-k sampling vs. greedy vs. temperature
print("=== Sampling Strategies on GPT-medium ===")
seed = torch.tensor([[char2idx['T']]], device=device)

# Greedy (temperature=0 effectively, top-k=1)
# We simulate by using very low temperature
print("\nGreedy (temp=0.1):")
out = gpt_medium.generate(seed, max_new_tokens=200, temperature=0.1, top_k=1)
print(decode(out))

print("\nTop-k=5, temp=0.8:")
out = gpt_medium.generate(seed, max_new_tokens=200, temperature=0.8, top_k=5)
print(decode(out))

print("\nTop-k=None (all), temp=1.0:")
out = gpt_medium.generate(seed, max_new_tokens=200, temperature=1.0, top_k=None)
print(decode(out))

## Key Takeaways

| Concept | What we learned |
|---|---|
| **Causal masking** | Upper-triangular mask prevents attending to future tokens |
| **Decoder-only** | No cross-attention — all attention is self-attention with causal mask |
| **Pre-LN** | Normalize before sub-layers — more stable training |
| **GELU** | Smoother than ReLU; used in GPT-2 and later |
| **Weight tying** | Output projection = input embedding matrix; reduces parameters |
| **Learned PE** | GPT uses learned positional embeddings, not sinusoidal |
| **Top-k sampling** | Restricts distribution to top-k tokens; avoids degenerate low-prob tokens |
| **Scale vs. loss** | More parameters → lower loss on same data |

**What GPT is not**: GPT is unidirectional — it only sees past tokens.
For tasks like sentiment classification or named entity recognition, you want
bidirectional context — seeing the full sentence before classifying any token.

Next: **BERT** — bidirectional Transformer trained with masked language modeling
and a fine-tuning recipe for downstream tasks.